# AeroPure — Weeks 3 & 4: Linear Regression (OLS Baseline), Ridge & Lasso Regression

### 1. Overview
In Weeks 3 and 4, we establish our foundational linear regression baselines to predict tomorrow's AQI:
- **Linear Regression (OLS Baseline)**: Minimizes $\sum (y_i - \hat{y}_i)^2$. Serves as the transparent reference.
- **Ridge Regression (L2)**: Minimizes RSS $+ \alpha \sum \beta_j^2$. Shrinks collinear coefficients.
- **Lasso Regression (L1)**: Minimizes RSS $+ \alpha \sum |\beta_j|$. Enforces sparsity and automatic feature selection.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(".."))
from src.regression import train_ols_regression, train_ridge_regression, train_lasso_regression
from src.evaluation import plot_actual_vs_predicted, plot_residuals

# Load preprocessed data if available
PROC_PATH = os.path.join("..", "data", "processed_data.csv")
if os.path.exists(PROC_PATH):
    from src.feature_engineering import prepare_time_series_splits
    df_proc = pd.read_csv(PROC_PATH)
    (X_train, X_test, y_train_reg, y_test_reg, _, _, _, feature_cols) = prepare_time_series_splits(df_proc)
    print(f"Train: {X_train.shape}, Test: {X_test.shape}")
else:
    print("Processed dataset not found. Run pipeline or Week 2 notebook first.")


### 2. OLS Linear Regression Baseline
Fitting Ordinary Least Squares on scaled training features.


In [ ]:
if "X_train" in locals():
    ols_model, ols_metrics, ols_preds, ols_coefs = train_ols_regression(X_train, y_train_reg, X_test, y_test_reg)
    print("OLS Baseline Metrics:", ols_metrics)
    plt.figure(figsize=(8, 6), dpi=120)
    plt.scatter(y_test_reg, ols_preds, alpha=0.4, color="#2b5c8f")
    plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], 'r--')
    plt.xlabel("Actual Next-Day AQI")
    plt.ylabel("Predicted Next-Day AQI")
    plt.title("OLS Linear Regression: Actual vs Predicted")
    plt.show()


### 3. Ridge Regression (L2 Penalty)
Mitigating multi-collinearity between correlated lags and rolling pollutant averages.


In [ ]:
if "X_train" in locals():
    ridge_model, ridge_metrics, ridge_preds, ridge_coefs = train_ridge_regression(X_train, y_train_reg, X_test, y_test_reg, alpha=10.0)
    print("Ridge Metrics:", ridge_metrics)


### 4. Lasso Regression (L1 Penalty)
Inducing coefficient sparsity to identify the most critical linear predictors.


In [ ]:
if "X_train" in locals():
    lasso_model, lasso_metrics, lasso_preds, lasso_coefs = train_lasso_regression(X_train, y_train_reg, X_test, y_test_reg, alpha=0.5)
    print("Lasso Metrics:", lasso_metrics)
    non_zero = (lasso_coefs != 0).sum()
    print(f"Lasso retained {non_zero} non-zero features out of {len(lasso_coefs)}")


### 5. Coefficient Comparison
Comparing feature coefficients across OLS, Ridge, and Lasso models.


In [ ]:
if "X_train" in locals():
    coef_comp = pd.DataFrame({
        "OLS": ols_coefs,
        "Ridge": ridge_coefs,
        "Lasso": lasso_coefs
    })
    display(coef_comp.head(15))


### Week 3 & 4 Summary
- OLS established an honest linear baseline for next-day AQI.
- Ridge stabilized collinear feature weights.
- Lasso zeroed out redundant noise features while preserving primary lag drivers.
